In [6]:
import pandas as pd

# 1. Carga
df_mayor = pd.read_csv("../data/raw/libro_mayor.csv")
df_banco = pd.read_csv("../data/raw/extracto_bancario.csv")

# 2. Convertimos fechas para que no haya errores de tipo
df_mayor['fecha'] = pd.to_datetime(df_mayor['fecha'])
df_banco['fecha'] = pd.to_datetime(df_banco['fecha'])

# 3. Nivel 1: Match Exacto
# reset_index() crea una columna llamada 'index' para saber de dónde venía cada fila
match_exacto = pd.merge(
    df_mayor.reset_index(), 
    df_banco.reset_index(), 
    on=['fecha', 'monto'], 
    how='inner', 
    suffixes=('_mayor', '_banco')
)

print(f"✅ Nivel 1 terminado. Movimientos conciliados: {len(match_exacto)}")
# Verificamos nombres de columnas para que el Nivel 2 no falle
print(f"Columnas creadas: {match_exacto.columns.tolist()}")

✅ Nivel 1 terminado. Movimientos conciliados: 263
Columnas creadas: ['index_mayor', 'fecha', 'monto', 'descripcion_mayor', 'index_banco', 'descripcion_banco']


In [7]:
# 1. Filtrar lo que ya conciliamos (usando los índices guardados)
df_mayor_no_match = df_mayor.loc[~df_mayor.index.isin(match_exacto['index_mayor'])].copy()
df_banco_no_match = df_banco.loc[~df_banco.index.isin(match_exacto['index_banco'])].copy()

# 2. Ordenar (REQUISITO para merge_asof: 'fecha' debe quedar globalmente ordenada)
df_mayor_no_match = df_mayor_no_match.sort_values(['fecha', 'monto']).reset_index(drop=True)
df_banco_no_match = df_banco_no_match.sort_values(['fecha', 'monto']).reset_index(drop=True)

# 3. Nivel 2: El Motor de Cercanía (Clearing de 3 días)
match_asof_3d = pd.merge_asof(
    df_mayor_no_match,
    df_banco_no_match,
    on="fecha",
    by="monto",
    tolerance=pd.Timedelta(days=3),
    direction="forward",
    suffixes=("_mayor", "_banco")
)

# 4. Resultados reales del Nivel 2 (quitando los que quedaron vacíos)
conciliados_nivel_2 = match_asof_3d.dropna(subset=['descripcion_banco'])

print(f"✅ Nivel 2 terminado.")
print(f"Movimientos rescatados por clearing: {len(conciliados_nivel_2)}")
match_asof_3d

✅ Nivel 2 terminado.
Movimientos rescatados por clearing: 736


,fecha,monto,descripcion_mayor,descripcion_banco
0,2026-03-01,-74068.84,Pago - Proveedor_5,BCO_MOV_PROV_REF680
1,2026-03-01,-64158.12,Pago - Aguas Cordobesas,BCO_MOV_AGUA_REF97
2,2026-03-01,-54937.94,Pago - Ecogas,BCO_MOV_ECOG_REF970
3,2026-03-01,-53693.50,Pago - Proveedor_7,BCO_MOV_PROV_REF958
4,2026-03-01,-48309.97,Pago - Proveedor_4,BCO_MOV_PROV_REF957
...,...,...,...,...
732,2026-03-25,60652.76,Cobro - Cliente_48,BCO_MOV_CLIE_REF226
733,2026-03-25,64998.56,Cobro - Cliente_40,BCO_MOV_CLIE_REF864
734,2026-03-25,93013.24,Cobro - Cliente_24,BCO_MOV_CLIE_REF833
735,2026-03-25,96991.16,Cobro - Cliente_49,BCO_MOV_CLIE_REF1000


In [12]:
import os

# --- EL CIERRE: REPORTE DE EXCEPCIONES (PARTIDAS PENDIENTES) ---

# 1. Pendientes del Libro Mayor
# Filtramos los que en el Nivel 2 no encontraron pareja en el banco (tienen NaN)
pendientes_mayor = match_asof_3d[match_asof_3d['descripcion_banco'].isna()].copy()
# Nos quedamos solo con las columnas del Mayor para que el Excel quede limpio
pendientes_mayor = pendientes_mayor[['fecha', 'monto', 'descripcion_mayor']]

# 2. Pendientes del Banco
# De todos los movimientos del banco sin matchear, quitamos los que ya cruzamos con éxito en Nivel 2
banco_usados_nivel2 = conciliados_nivel_2['descripcion_banco'].tolist()
pendientes_banco = df_banco_no_match[
    ~df_banco_no_match['descripcion'].isin(banco_usados_nivel2)
].copy()

print("-" * 50)
print("📊 RESUMEN DE EXCEPCIONES PARA AUDITORÍA MANUAL:")
print(f"⚠️ En Libro Mayor (Posibles errores o no procesados): {len(pendientes_mayor)}")
print(f"⚠️ En Banco (Comisiones, impuestos o cobros no registrados): {len(pendientes_banco)}")
print("-" * 50)

# 3. Preparamos la carpeta de salida
ruta_carpeta = "../data/processed"
os.makedirs(ruta_carpeta, exist_ok=True)
ruta_archivo = f"{ruta_carpeta}/Partidas_Pendientes_Marzo.xlsx"

# 4. Exportamos a Excel
try:
    with pd.ExcelWriter(ruta_archivo, engine='openpyxl') as writer:
        pendientes_mayor.to_excel(writer, sheet_name='Revisar_en_Mayor', index=False)
        pendientes_banco.to_excel(writer, sheet_name='Revisar_en_Banco', index=False)
    print(f"✅ ¡Reporte Excel generado con éxito en: {ruta_archivo}!")
except ModuleNotFoundError:
    print("❌ Falta el motor de Excel. Abrí tu terminal Git Bash y ejecutá: pip install openpyxl")

--------------------------------------------------
📊 RESUMEN DE EXCEPCIONES PARA AUDITORÍA MANUAL:
⚠️ En Libro Mayor (Posibles errores o no procesados): 1
⚠️ En Banco (Comisiones, impuestos o cobros no registrados): 2
--------------------------------------------------
✅ ¡Reporte Excel generado con éxito en: ../data/processed/Partidas_Pendientes_Marzo.xlsx!
